# Working within a factor of the best algorithm. 

For this part, I will use the data that I have already collected regarding the algo that has the best ERT. 

This startegy is a smart approach to keeping in mind algorithms that behave similarly. This is method is beneficial for algorithms that have are slightly less performing than the best algorithm, but still remain interesting to investigate their performance. This contrasts the other method where only the best algo was kept and all other algorithms were discarded. 
## 1) Comparing the best ERT of each dimension, function, target with the algos from all the different years. 

In this part, I will read the bestERT from the file global_best_algos.csv. This bestERT is determinant in the definition of the best algorithms and will determine which algos I can keep. 

We will define a factor (ex: 2), and we will keep all the algorithms that have an ERT that is within 2 times the ERT of the overall bestERT. This is done over every dimension, function and target.  

The output result of this will give us a CSV file tracking which are these good algorithms for each dimension, function and target precision. 

These are the following steps I did to implement the code:

1) Read the csv file where the best algos appear, and all the information regarding dimension, function and target 
2) Loop over every year. 
3) Create an interactive interface where we ask the user what factor they want. 
4) Inside every year, loop over dimension function and target and check if the ERT is within the factor bounds. 
5) If it is: write the name of the algo, the year, the dimension, the function and the target to a CSV file. 
6) If it isn't then pass and keep going to the next algo.

In [7]:
import pandas as pd
import numpy as np
import cocopp


# ============================================================
# 1. LOAD YOUR CSV WITH BEST ERTs  
# ============================================================

# make sure to change the file path accordding to what test suite you are working with 

df_best = pd.read_csv("results/global_best_algos_bbob-biobj.csv")

# clean
df_best = df_best[df_best.best_algorithm.notna()]
df_best = df_best[df_best.best_ERT.notna()]

# all known targets (sorted, unique)
all_targets = sorted(df_best["target"].unique())

# dictionary: (dim, func, target) → best_ERT
best_dict = {}
for _, row in df_best.iterrows():
    key = (int(row.dimension), int(row.function_id), float(row.target))
    best_dict[key] = float(row.best_ERT)


# ============================================================
# 2. ASK USER FOR FACTOR
# ============================================================

try:
    factor = float(input("Enter tolerance factor (default=2): ") or 2)
except:
    factor = 2.0

print(f"\nUsing factor = {factor}\n")


# ============================================================
# 3. LOAD COCO DATA BY YEAR
# ============================================================
# again make sure to select the right file path for the corresponding test suite 

years = df_best["year"].unique()     # automatically detect existing years
years = sorted(years)

dsl_by_year = {}

for y in years:
    print(f"Loading BBOB noisy data for year {y} ...")
    try:
        dsl_by_year[y] = cocopp.load(f"bbob-biobj/{y}/*")
    except:
        print(f" Warning: Could not load year {y}. Skipping.")
        continue


# ============================================================
# 4. LOOP USING YOUR EXACT CODE STRUCTURE
# ============================================================

within_rows = []

for year, dsl in dsl_by_year.items():
    
    print(f"\n=== Processing year {year} ===")
    
    dd = dsl.dictByDimFunc()     # same as in your working code

    for dim in sorted(dd.keys()):
        for func in sorted(dd[dim].keys()):
            
            # loop over algorithms
            for ds in dd[dim][func]:
                
                algo = ds.algId

                # loop over targets from your CSV
                for tgt in all_targets:

                    key = (dim, func, tgt)
                    if key not in best_dict:
                        continue     # best ERT not known for this triple

                    best_ert = best_dict[key]

                    # compute ERT for THIS algorithm at THIS target
                    try:
                        ert = float(ds.detERT([tgt])[0])
                    except Exception:
                        continue

                    if not np.isfinite(ert):
                        continue

                    # check whether ERT is within factor
                    if ert <= factor * best_ert:
                        within_rows.append([
                            year,
                            dim,
                            func,
                            tgt,
                            algo,
                            ert,
                            best_ert
                        ])


# ============================================================
# 5. SAVE OUTPUT CSV
# ============================================================

within_df = pd.DataFrame(
    within_rows,
    columns=[
        "year",
        "dimension",
        "function_id",
        "target",
        "algorithm",
        "ERT",
        "best_ERT"
    ]
)

output_path = "results/within_factor_algorithms_bbob-biobj.csv"
within_df.to_csv(output_path, index=False)

print(f"\n✓ Saved: {output_path}")
print(f"Total rows: {len(within_df)}")


Enter tolerance factor (default=2): 2

Using factor = 2.0

Loading BBOB noisy data for year 2016 ...
Loading BBOB noisy data for year 2019 ...
Loading BBOB noisy data for year 2021 ...
Loading BBOB noisy data for year 2022 ...

=== Processing year 2016 ===

=== Processing year 2019 ===

=== Processing year 2021 ===

=== Processing year 2022 ===

✓ Saved: results/within_factor_algorithms_bbob-biobj.csv
Total rows: 3768


## 2) Counting the number of times an algo appears for each dimension

In this part, we are going to be aggregating over all the different function and targets within a certain dimension. The goal is to have for each dimension the best performing algorithms with the factor determined above, and also display the count.

In [8]:
import pandas as pd

# ============================================================
# 1. READ THE CSV FROM PART 1
# ============================================================

df = pd.read_csv("results/within_factor_algorithms_bbob-biobj.csv")

# clean potential NaNs
df = df[df.algorithm.notna()]
df = df[df.ERT.notna()]


# ============================================================
# 2. INITIALIZE THE COUNTER
# ============================================================
# We use a dictionary: (dimension, algorithm) → count

counter = {}

for _, row in df.iterrows():
    dim  = int(row["dimension"])
    algo = row["algorithm"]

    key = (dim, algo)

    if key not in counter:
        counter[key] = 0
    counter[key] += 1


# ============================================================
# 3. CONVERT DICTIONARY TO DATAFRAME
# ============================================================

rows = []
for (dim, algo), count in counter.items():
    rows.append([dim, algo, count])

counts_df = pd.DataFrame(rows, columns=["dimension", "algorithm", "count"])


# Sort: first by dimension, then by descending count
counts_df = counts_df.sort_values(["dimension", "count"], ascending=[True, False])


# ============================================================
# 4. SAVE THE RESULTING CSV
# ============================================================

output_path = "results/counts_by_dimension_bbob-biobj.csv"
counts_df.to_csv(output_path, index=False)

print(f"\n✓ Saved: {output_path}")
print(f"Total rows: {len(counts_df)}")

# Optional: display a preview
print("\n=== Preview ===")
print(counts_df.head(20))



✓ Saved: results/counts_by_dimension_bbob-biobj.csv
Total rows: 142

=== Preview ===
     dimension                                 algorithm  count
1            2          HMO-CMA-ES_Loshchilov_bbob-biobj    125
2            2         MO-DIRECT-HV-Rank_Wong_bbob-biobj     74
6            2            UP-MO-CMA-ES_Krause_bbob-biobj     68
0            2                     DEMO_Tusar_bbob-biobj     63
56           2       SPEA2-platypus_Brockhoff_bbob-biobj     61
128          2                             K-RVEA_Tanabe     57
47           2  MO-CMA-ES-100-autoref_Dufosse_bbob-biobj     53
46           2        IBEA-platypus_Brockhoff_bbob-biobj     51
48           2   MO-CMA-ES-32-autoref_Dufosse_bbob-biobj     50
7            2                  RM-MEDA_Auger_bbob-biobj     45
50           2        GDE3-platypus_Brockhoff_bbob-biobj     44
130          2                              MOTPE_Tanabe     43
53           2     NSGA-II-platypus_Brockhoff_bbob-biobj     41
10           2    

In [9]:
import pandas as pd

# ============================================================
# 1. LOAD THE COUNTS DATA
# ============================================================

df = pd.read_csv("results/counts_by_dimension_bbob-biobj.csv")

# Clean and sort
df = df.reset_index(drop=True)
df = df.sort_values(["dimension", "count"], ascending=[True, False])


# ============================================================
# 2. BUILD DICTIONARY: dim → list of (algo, count)
# ============================================================

ranking_dict = {}

for dim in sorted(df["dimension"].unique()):
    df_dim = df[df["dimension"] == dim]
    ranking_dict[dim] = list(zip(df_dim["algorithm"], df_dim["count"]))


# ============================================================
# 3. DETERMINE MAX RANK
# ============================================================

max_len = max(len(v) for v in ranking_dict.values())


# ============================================================
# 4. BUILD WIDE RANKING TABLE
# ============================================================

rows = []

for rank in range(max_len):
    row = {"rank": rank + 1}

    for dim in sorted(ranking_dict.keys()):
        if rank < len(ranking_dict[dim]):
            algo, count = ranking_dict[dim][rank]
            row[f"dim {dim}"] = f"{algo} ({count})"
        else:
            row[f"dim {dim}"] = ""
    
    rows.append(row)

ranking_table = pd.DataFrame(rows)


# ============================================================
# 5. DISPLAY NICELY USING PANDAS STYLING
# ============================================================

styled_table = (
    ranking_table.style
        .set_properties(**{
            "background-color": "#f7f7f7",
            "border": "1px solid #ccc",
            "padding": "6px",
            "font-size": "12px"
        })
        .set_table_styles([
            {"selector": "th", 
             "props": [("background-color", "#e6e6e6"),
                       ("font-weight", "bold"),
                       ("border", "1px solid #aaa"),
                       ("padding", "6px")]}
        ])
        .hide(axis="index")  # Hide the pandas index entirely
)

styled_table





rank,dim 2,dim 3,dim 5,dim 10,dim 20,dim 40
1,HMO-CMA-ES_Loshchilov_bbob-biobj (125),HMO-CMA-ES_Loshchilov_bbob-biobj (135),HMO-CMA-ES_Loshchilov_bbob-biobj (141),HMO-CMA-ES_Loshchilov_bbob-biobj (152),HMO-CMA-ES_Loshchilov_bbob-biobj (159),GDE3-platypus_Brockhoff_bbob-biobj (130)
2,MO-DIRECT-HV-Rank_Wong_bbob-biobj (74),SPEA2-platypus_Brockhoff_bbob-biobj (55),UP-MO-CMA-ES_Krause_bbob-biobj (48),TPB_Tanabe (45),TPB_Tanabe (42),RM-MEDA_Auger_bbob-biobj (96)
3,UP-MO-CMA-ES_Krause_bbob-biobj (68),K-RVEA_Tanabe (53),TPB_Tanabe (41),UP-MO-CMA-ES_Krause_bbob-biobj (38),UP-MO-CMA-ES_Krause_bbob-biobj (34),SPEA2-platypus_Brockhoff_bbob-biobj (62)
4,DEMO_Tusar_bbob-biobj (63),UP-MO-CMA-ES_Krause_bbob-biobj (49),SPEA2-platypus_Brockhoff_bbob-biobj (39),GDE3-platypus_Brockhoff_bbob-biobj (38),GDE3-platypus_Brockhoff_bbob-biobj (31),MOEAD-platypus_Brockhoff_bbob-biobj (24)
5,SPEA2-platypus_Brockhoff_bbob-biobj (61),TPB_Tanabe (46),K-RVEA_Tanabe (37),COMO-316_dufosse_bbob-biobj (32),SPEA2-platypus_Brockhoff_bbob-biobj (23),SMS-EMOA-DE_Auger_bbob-biobj (1)
6,K-RVEA_Tanabe (57),MO-DIRECT-HV-Rank_Wong_bbob-biobj (41),GDE3-platypus_Brockhoff_bbob-biobj (32),IBEA-platypus_Brockhoff_bbob-biobj (27),COMO-316_dufosse_bbob-biobj (19),
7,MO-CMA-ES-100-autoref_Dufosse_bbob-biobj (53),RM-MEDA_Auger_bbob-biobj (40),IBEA-platypus_Brockhoff_bbob-biobj (28),SPEA2-platypus_Brockhoff_bbob-biobj (25),IBEA-platypus_Brockhoff_bbob-biobj (18),
8,IBEA-platypus_Brockhoff_bbob-biobj (51),MO-CMA-ES-100-autoref_Dufosse_bbob-biobj (37),MOTPE_Tanabe (27),COMO-100_dufosse_bbob-biobj (17),COMO-100_dufosse_bbob-biobj (13),
9,MO-CMA-ES-32-autoref_Dufosse_bbob-biobj (50),GDE3-platypus_Brockhoff_bbob-biobj (36),COMO-316_dufosse_bbob-biobj (26),K-RVEA_Tanabe (16),COMO-10_dufosse_bbob-biobj (8),
10,RM-MEDA_Auger_bbob-biobj (45),MOTPE_Tanabe (36),COMO-100_dufosse_bbob-biobj (24),MO-CMA-ES-100-autoref_Dufosse_bbob-biobj (13),RM-MEDA_Auger_bbob-biobj (6),


An interesting result is that the algo IPOPsaACM_loshchilov_noisy dominates in performance in almost all categories, except the last one (dim 40), where it is IPOP-ACTCMA-ES_ros_noisy

In [10]:
# Aggregate counts over all dimensions
overall_counts = (
    df.groupby("algorithm")["count"]
      .sum()
      .reset_index()
      .sort_values("count", ascending=False)
)

print("\nOverall frequency of being best (aggregated over ALL dimensions):\n")
print(overall_counts.head(10))



Overall frequency of being best (aggregated over ALL dimensions):

                                   algorithm  count
9           HMO-CMA-ES_Loshchilov_bbob-biobj    712
8         GDE3-platypus_Brockhoff_bbob-biobj    311
30       SPEA2-platypus_Brockhoff_bbob-biobj    265
32            UP-MO-CMA-ES_Krause_bbob-biobj    237
27                  RM-MEDA_Auger_bbob-biobj    216
31                                TPB_Tanabe    211
11                             K-RVEA_Tanabe    164
10        IBEA-platypus_Brockhoff_bbob-biobj    159
13  MO-CMA-ES-100-autoref_Dufosse_bbob-biobj    126
15         MO-DIRECT-HV-Rank_Wong_bbob-biobj    126


In [11]:
overall_counts["percentage"] = (
    overall_counts["count"] / overall_counts["count"].sum() * 100
).round(2)

overall_counts.head(10)


,algorithm,count,percentage
9,HMO-CMA-ES_Loshchilov_bbob-biobj,712,18.90
8,GDE3-platypus_Brockhoff_bbob-biobj,311,8.25
30,SPEA2-platypus_Brockhoff_bbob-biobj,265,7.03
32,UP-MO-CMA-ES_Krause_bbob-biobj,237,6.29
27,RM-MEDA_Auger_bbob-biobj,216,5.73
31,TPB_Tanabe,211,5.60
11,K-RVEA_Tanabe,164,4.35
10,IBEA-platypus_Brockhoff_bbob-biobj,159,4.22
13,MO-CMA-ES-100-autoref_Dufosse_bbob-biobj,126,3.34
15,MO-DIRECT-HV-Rank_Wong_bbob-biobj,126,3.34


In [13]:
import pandas as pd

# ============================================================
# 1. LOAD THE WITHIN-FACTOR DATA
# ============================================================

df = pd.read_csv("results/within_factor_algorithms_bbob-biobj.csv")

# Ensure clean
df = df[df.algorithm.notna()]
df = df[df.ERT.notna()]


# ============================================================
# 2. FUNCTION TO BUILD A RANKING TABLE FOR A GIVEN TARGET
# ============================================================

def make_dimension_ranking_table_with_counts(df, target):
    """
    For a given target:
    - Count how many times each algorithm is within factor for each dimension.
    - Rank algorithms per dimension by this count.
    - Build a pivot table: rows = rank, columns = dimension, values = (algo, count).
    """
    df_t = df[df["target"] == target].copy()
    if df_t.empty:
        raise ValueError(f"No data found for target = {target}")

    algo_counts = (
        df_t.groupby(["dimension", "algorithm"])
        .size()
        .reset_index(name="count")
    )

    algo_counts = algo_counts.sort_values(["dimension", "count"], ascending=[True, False])

    algo_counts["rank"] = (
        algo_counts.groupby("dimension")["count"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    algo_counts["algo_tuple"] = list(zip(algo_counts["algorithm"], algo_counts["count"]))

    ranking_table = algo_counts.pivot(index="rank", columns="dimension", values="algo_tuple")

    ranking_table = ranking_table.reindex(sorted(ranking_table.columns), axis=1)

    return ranking_table


# ============================================================
# 3. INTERACTIVE: ASK THE USER FOR A TARGET
# ============================================================

try:
    print("Available targets:", sorted(df["target"].unique()))

    target_input = float(input("\nSelect a target precision (e.g., 1e-4, 1e-2, 1e-6): "))

    algo_ranking_table = make_dimension_ranking_table_with_counts(df, target_input)

    print(f"\nRanking table for target = {target_input}")
    from IPython.display import display
    display(algo_ranking_table)

except ValueError:
    print("Invalid input or no data for this target. Try a target from the list above.")


Available targets: [1e-08, 1e-05, 0.001, 0.01, 0.1]

Select a target precision (e.g., 1e-4, 1e-2, 1e-6): 1e-05

Ranking table for target = 1e-05


dimension,2,3,5,10,20,40
rank,,,,,,
1,"(HMO-CMA-ES_Loshchilov_bbob-biobj, 42)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 41)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 31)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 29)","(HMO-CMA-ES_Loshchilov_bbob-biobj, 32)","(GDE3-platypus_Brockhoff_bbob-biobj, 14)"
2,"(UP-MO-CMA-ES_Krause_bbob-biobj, 35)","(UP-MO-CMA-ES_Krause_bbob-biobj, 28)","(UP-MO-CMA-ES_Krause_bbob-biobj, 28)","(UP-MO-CMA-ES_Krause_bbob-biobj, 21)","(UP-MO-CMA-ES_Krause_bbob-biobj, 20)","(MOEAD-platypus_Brockhoff_bbob-biobj, 2)"
3,"(MO-CMA-ES-100-autoref_Dufosse_bbob-biobj, 19)","(MO-CMA-ES-100-autoref_Dufosse_bbob-biobj, 5)","(COMO-316_dufosse_bbob-biobj, 5)","(COMO-316_dufosse_bbob-biobj, 6)","(COMO-100_dufosse_bbob-biobj, 3)","(RM-MEDA_Auger_bbob-biobj, 1)"
4,"(SPEA2-platypus_Brockhoff_bbob-biobj, 5)","(COMO-316_dufosse_bbob-biobj, 4)","(NSGA-II-platypus_Brockhoff_bbob-biobj, 4)","(COMO-100_dufosse_bbob-biobj, 4)","(COMO-316_dufosse_bbob-biobj, 2)","(SPEA2-platypus_Brockhoff_bbob-biobj, 1)"
5,"(MO-DIRECT-HV-Rank_Wong_bbob-biobj, 4)","(SPEA2-platypus_Brockhoff_bbob-biobj, 3)","(COMO-100_dufosse_bbob-biobj, 3)","(DEMO_Tusar_bbob-biobj, 1)","(GDE3-platypus_Brockhoff_bbob-biobj, 1)",NaN
6,"(COMO-316_dufosse_bbob-biobj, 3)","(MO-CMA-ES-32-autoref_Dufosse_bbob-biobj, 2)","(COMO-1e3_dufosse_bbob-biobj, 1)","(GDE3-platypus_Brockhoff_bbob-biobj, 1)",NaN,NaN
7,"(IBEA-platypus_Brockhoff_bbob-biobj, 3)","(COMO-100_dufosse_bbob-biobj, 1)","(DEMO_Tusar_bbob-biobj, 1)","(IBEA-platypus_Brockhoff_bbob-biobj, 1)",NaN,NaN
8,"(MO-DIRECT-Rank_Wong_bbob-biobj, 3)","(COMO-1e3_dufosse_bbob-biobj, 1)","(DMS_Brockhoff_bbob-biobj, 1)","(NSGA-II-platypus_Brockhoff_bbob-biobj, 1)",NaN,NaN
9,"(RM-MEDA_Auger_bbob-biobj, 3)","(DEMO_Tusar_bbob-biobj, 1)","(GDE3-platypus_Brockhoff_bbob-biobj, 1)","(SMS-EMOA-DE_Auger_bbob-biobj, 1)",NaN,NaN


# 3) Plotting the results using the COCO platform tool

This last step is essential for checking if the result sin our table are indeed coherent. 
Using the COCO tool, we can access make a full analysis of the best performing algorithms. 

## 3.1) Performance of the best algorithms for each dimension

The first comparison between algorithms I want to do, is see the performance of the best algorithms for the different dimensionsion. There is a clear dominance of one algorihm as it appears as best in 5/6 dimensions.

On a side note, these 2 algos are also the best 2 for the dimension 2.

In [7]:
cocopp.main(['IPOPsaACM_loshchilov_noisy','IPOP-ACTCMA-ES_ros_noisy'])

Post-processing (2+)
  downloading https://numbbo.github.io/data-archive/data-archive/bbob-noisy/2012/IPOPsaACM_loshchilov_noisy.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz
  Using 2 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-ACTCMA-ES_ros_noisy.tar.gz

Post-processing (2+)
  loading data...
    archive extracted to folder C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\.extracted_IPOPsaACM_loshchilov_noisy ...
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2010\IPOP-ACTCMA-ES_ros_noisy.tar.gz
  Will generate output data in folder ppdata\I

DictAlg([(('IPOPsaACM_loshchilov_noisy', ''),
          [DataSet(IPOPsaACM_loshchilov_noisy on f101 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f102 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f103 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f104 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f105 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f106 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f107 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f108 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f109 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f110 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f111 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f112 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f113 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f114 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f115 2-D),
           DataSet(IPOPsaACM_loshchilov_noisy on f116 

In [14]:
cocopp.main(['HMO-CMA-ES_Loshchilov_bbob-biobj','GDE3-platypus_Brockhoff_bbob-biobj','SPEA2-platypus_Brockhoff_bbob-biobj','UP-MO-CMA-ES_Krause_bbob-biobj','RM-MEDA_Auger_bbob-biobj'])

Post-processing (2+)
  Using 5 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\SPEA2-platypus_Brockhoff_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\UP-MO-CMA-ES_Krause_bbob-biobj.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\RM-MEDA_Auger_bbob-biobj.tgz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\HMO-CMA-ES_Loshchilov_bbob-biobj.tgz


C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pproc.py:3895: UserWarning:  Reference values for the algorithm 'C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz' are different!
  warnings.warn(" Reference values for the algorithm '%s' are different!" % alg)


  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\GDE3-platypus_Brockhoff_bbob-biobj.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2019\SPEA2-platypus_Brockhoff_bbob-biobj.tgz


C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\pproc.py:3895: UserWarning:  Reference values for the algorithm 'C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\UP-MO-CMA-ES_Krause_bbob-biobj.tgz' are different!
  warnings.warn(" Reference values for the algorithm '%s' are different!" % alg)


  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\UP-MO-CMA-ES_Krause_bbob-biobj.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-biobj\2016\RM-MEDA_Auger_bbob-biobj.tgz


C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsettings.py:126: UserWarning:  Reference values for the algorithm 'GDE3-platypus_Brockhoff_bbob-biobj' are different from the algorithm 'HMO-CMA-ES_Loshchilov_bbob-biobj'
  warnings.warn(" Reference values for the algorithm '%s' are different from the algorithm '%s'"
C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\testbedsettings.py:126: UserWarning:  Reference values for the algorithm 'SPEA2-platypus_Brockhoff_bbob-biobj' are different from the algorithm 'HMO-CMA-ES_Loshchilov_bbob-biobj'
  warnings.warn(" Reference values for the algorithm '%s' are different from the algorithm '%s'"


  Will generate output data in folder ppdata\biobj_HMO-C_GDE3-_SPEA2_UP-MO_RM-ME_112817h4350
    this might take several minutes.
ECDF graphs per noise group...
Loading best algorithm data from refalgs/best2016-bbob-biobj.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2016-bbob-biobj.tar.gz
  done (Fri Nov 28 17:43:54 2025).
  done (Fri Nov 28 17:44:08 2025).
ECDF graphs per function group...
  done (Fri Nov 28 17:45:40 2025).
ECDF graphs per function...
  done (Fri Nov 28 17:52:26 2025).
Generating comparison tables...
  done (Fri Nov 28 17:53:29 2025).
Scaling figures...
  done (Fri Nov 28 20:30:13 2025).
Output data written to folder C:\Users\elsaf\PRL project on jupyter\ppdata\biobj_HMO-C_GDE3-_SPEA2_UP-MO_RM-ME_112817h4350
Setting changes in `cocopp.genericsettings` compared to default:
    simulated_runlength_bootstrap_sample_size: from 30 to 10.098990100989901
    foreground_algorithm_list: from [] to ['C:\\Users\\elsaf\\Ap...
ALL done (Fri Nov

DictAlg([(('HMO-CMA-ES_Loshchilov_bbob-biobj', ''),
          [DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f1 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f2 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f3 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f4 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f5 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f6 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f7 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f8 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f9 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f10 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f11 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f12 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f13 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-biobj on f14 2-D),
           DataSet(HMO-CMA-ES_Loshchilov_bbob-b

If this type of error occurs:

Exception: There is more than a single entry associated with folder C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob-noisy\2012\IPOPsaACM_loshchilov_noisy.tgz on 2-D f101.


--> Go to the folder and delete it. 
--> When run the cocopp platform it will reinstall it.

We notice from the plots that these algorithms have a very similar behavior and almost overlap at every instance. 


## 3.2) Comparison of the 3 best algorithms for all dimensions

Here We observe that in almost all dimensions, the leaderboard for the top 3 algorithms is the same accross all dimensions except the last dimension 40. 


In [ ]:
cocopp.main(['IPOPsaACM_loshchilov_noisy','IPOP-ACTCMA-ES_ros_noisy','IPOP-CMA-ES_ros_noisy'])